# Preprocessing Dataset Mahasiswa — Data Real 2020/2021
Pipeline: Load Data → Seleksi Atribut → Cleaning → Discretization → Labeling Risiko → Simpan

## 1. Eksplorasi Awal

In [3]:
import pandas as pd

# Load dataset real (.xls)
df = pd.read_excel('dataset/raw/data_mhs_2020_2021_new.xls', engine='xlrd', dtype={'nim': str})
print(f'Jumlah data: {df.shape[0]} baris, {df.shape[1]} kolom')
df.info()

Jumlah data: 55 baris, 14 kolom
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   nim                              55 non-null     object 
 1   jumlah_semester                  55 non-null     int64  
 2   ipk                              55 non-null     float64
 3   ips                              36 non-null     float64
 4   total_sks_lulus                  55 non-null     int64  
 5   jumlah_mk_diulang                55 non-null     int64  
 6   semester_terakhir_aktif          55 non-null     int64  
 7   semester_tidak_aktif_berturut    55 non-null     int64  
 8   jml_mk_wajib                     55 non-null     int64  
 9   jml_mk_pilihan_bebas_non_bidang  55 non-null     int64  
 10  jml_mk_pilihan_bebas_prodi       55 non-null     int64  
 11  jml_mk_pilihan_wajib_profil      55 non-null     int64

In [4]:
df.describe()

,jumlah_semester,ipk,ips,total_sks_lulus,jumlah_mk_diulang,semester_terakhir_aktif,semester_tidak_aktif_berturut,jml_mk_wajib,jml_mk_pilihan_bebas_non_bidang,jml_mk_pilihan_bebas_prodi,jml_mk_pilihan_wajib_profil
count,55.000000,55.000000,36.000000,55.000000,55.000000,55.000000,55.000000,55.000000,55.000000,55.000000,55.000000
mean,9.545455,2.893455,1.869444,134.727273,7.490909,20250.109091,0.163636,34.872727,1.181818,6.345455,7.945455
std,0.977870,0.493683,1.293966,19.975406,5.134750,3.435505,0.660068,1.503755,0.795611,2.196109,2.778343
min,7.000000,1.550000,0.000000,65.000000,0.000000,20231.000000,0.000000,30.000000,0.000000,1.000000,0.000000
25%,9.000000,2.545000,0.825000,130.000000,4.000000,20251.000000,0.000000,34.000000,1.000000,5.000000,6.000000
50%,9.000000,2.960000,1.615000,141.000000,6.000000,20251.000000,0.000000,35.000000,1.000000,7.000000,8.000000
75%,10.500000,3.290000,3.275000,146.000000,10.500000,20251.000000,0.000000,36.000000,2.000000,8.000000,9.000000
max,11.000000,3.720000,4.000000,173.000000,19.000000,20251.000000,4.000000,37.000000,3.000000,11.000000,14.000000


In [5]:
# Cek nilai unik pada kolom kategorikal
print('Unik stmhs:', df['stmhs'].unique())
print('Unik status:', df['status'].unique())

Unik stmhs: ['AR']
Unik status: ['A' 'T']


In [6]:
# Cek missing values
print('Missing values per kolom:')
print(df.isnull().sum())

Missing values per kolom:
nim                                 0
jumlah_semester                     0
ipk                                 0
ips                                19
total_sks_lulus                     0
jumlah_mk_diulang                   0
semester_terakhir_aktif             0
semester_tidak_aktif_berturut       0
jml_mk_wajib                        0
jml_mk_pilihan_bebas_non_bidang     0
jml_mk_pilihan_bebas_prodi          0
jml_mk_pilihan_wajib_profil         0
stmhs                               0
status                              0
dtype: int64


## 2. Seleksi Atribut

Kolom yang tidak relevan untuk pemodelan risiko akademik dihapus:
- `nim` → identitas mahasiswa, bukan fitur
- `stmhs` → bernilai konstan ('AR'), tidak informatif
- `ips` → banyak missing value (IPS semester terakhir), digantikan oleh `ipk`

In [7]:
df = df.drop(columns=['nim', 'stmhs', 'ips'])
print(f'Kolom tersisa: {df.columns.tolist()}')
df.head()

Kolom tersisa: ['jumlah_semester', 'ipk', 'total_sks_lulus', 'jumlah_mk_diulang', 'semester_terakhir_aktif', 'semester_tidak_aktif_berturut', 'jml_mk_wajib', 'jml_mk_pilihan_bebas_non_bidang', 'jml_mk_pilihan_bebas_prodi', 'jml_mk_pilihan_wajib_profil', 'status']


,jumlah_semester,ipk,total_sks_lulus,jumlah_mk_diulang,semester_terakhir_aktif,semester_tidak_aktif_berturut,jml_mk_wajib,jml_mk_pilihan_bebas_non_bidang,jml_mk_pilihan_bebas_prodi,jml_mk_pilihan_wajib_profil,status
0,11,3.03,147,6,20251,0,35,2,9,7,A
1,11,2.93,134,5,20251,0,36,0,11,6,A
2,7,2.36,106,5,20231,4,32,2,3,6,T
3,11,2.33,94,17,20251,0,31,1,3,5,A
4,11,2.52,138,10,20251,0,34,0,4,12,A


## 3. Cleaning Data

In [8]:
# Hapus duplikat
sebelum = df.shape[0]
df = df.drop_duplicates()
print(f'Baris dihapus (duplikat): {sebelum - df.shape[0]}')

Baris dihapus (duplikat): 0


In [9]:
# Hapus baris dengan nilai kosong (jika ada pada kolom yang tersisa)
sebelum = df.shape[0]
df = df.dropna()
print(f'Baris dihapus (missing): {sebelum - df.shape[0]}')
print(f'Total data bersih: {df.shape[0]} baris')
df.info()

Baris dihapus (missing): 0
Total data bersih: 55 baris
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   jumlah_semester                  55 non-null     int64  
 1   ipk                              55 non-null     float64
 2   total_sks_lulus                  55 non-null     int64  
 3   jumlah_mk_diulang                55 non-null     int64  
 4   semester_terakhir_aktif          55 non-null     int64  
 5   semester_tidak_aktif_berturut    55 non-null     int64  
 6   jml_mk_wajib                     55 non-null     int64  
 7   jml_mk_pilihan_bebas_non_bidang  55 non-null     int64  
 8   jml_mk_pilihan_bebas_prodi       55 non-null     int64  
 9   jml_mk_pilihan_wajib_profil      55 non-null     int64  
 10  status                           55 non-null     object 
dtypes: float64(1), int64(9), object

## 4. Discretization (Kategorisasi Fitur Numerik)

Mengubah nilai numerik menjadi kategori berdasarkan threshold akademik.

In [10]:
# Kategorisasi IPK
def kategori_ipk(ipk):
    if ipk < 2.5:
        return 'Rendah'
    elif ipk <= 2.75:
        return 'Sedang'
    else:
        return 'Baik'

df['kategori_ipk'] = df['ipk'].apply(kategori_ipk)
print(df['kategori_ipk'].value_counts())

kategori_ipk
Baik      34
Rendah    11
Sedang    10
Name: count, dtype: int64


In [11]:
# Kategorisasi total SKS lulus
# Threshold: minimal 144 SKS untuk lulus S1 (sesuaikan jika berbeda di prodi)
df['kategori_sks'] = df['total_sks_lulus'].apply(lambda x: 'Kurang' if x < 144 else 'Cukup')
print(df['kategori_sks'].value_counts())

kategori_sks
Kurang    36
Cukup     19
Name: count, dtype: int64


In [12]:
# Kategorisasi MK wajib
# Threshold: sesuaikan jumlah MK wajib minimum prodi
df['kategori_mk_wajib'] = df['jml_mk_wajib'].apply(lambda x: 'Kurang' if x < 30 else 'Cukup')
print(df['kategori_mk_wajib'].value_counts())

kategori_mk_wajib
Cukup    55
Name: count, dtype: int64


In [13]:
# Kategorisasi MK pilihan bebas non-bidang
df['kategori_mk_bebas_nb'] = df['jml_mk_pilihan_bebas_non_bidang'].apply(lambda x: 'Kurang' if x < 2 else 'Cukup')
print(df['kategori_mk_bebas_nb'].value_counts())

kategori_mk_bebas_nb
Kurang    34
Cukup     21
Name: count, dtype: int64


In [14]:
# Kategorisasi MK pilihan bebas prodi
df['kategori_mk_bebas_prodi'] = df['jml_mk_pilihan_bebas_prodi'].apply(lambda x: 'Kurang' if x < 5 else 'Cukup')
print(df['kategori_mk_bebas_prodi'].value_counts())

kategori_mk_bebas_prodi
Cukup     44
Kurang    11
Name: count, dtype: int64


In [15]:
# Kategorisasi MK pilihan wajib profil
df['kategori_mk_wajib_profil'] = df['jml_mk_pilihan_wajib_profil'].apply(lambda x: 'Kurang' if x < 6 else 'Cukup')
print(df['kategori_mk_wajib_profil'].value_counts())

kategori_mk_wajib_profil
Cukup     45
Kurang    10
Name: count, dtype: int64


In [16]:
# Kategorisasi jumlah MK diulang
df['kategori_mk_diulang'] = df['jumlah_mk_diulang'].apply(lambda x: 'Banyak' if x > 5 else 'Normal')
print(df['kategori_mk_diulang'].value_counts())

kategori_mk_diulang
Banyak    31
Normal    24
Name: count, dtype: int64


In [17]:
# Lihat hasil discretization
df[['ipk','kategori_ipk','total_sks_lulus','kategori_sks',
    'jml_mk_wajib','kategori_mk_wajib','jumlah_mk_diulang','kategori_mk_diulang']].head(10)

,ipk,kategori_ipk,total_sks_lulus,kategori_sks,jml_mk_wajib,kategori_mk_wajib,jumlah_mk_diulang,kategori_mk_diulang
0,3.03,Baik,147,Cukup,35,Cukup,6,Banyak
1,2.93,Baik,134,Kurang,36,Cukup,5,Normal
2,2.36,Rendah,106,Kurang,32,Cukup,5,Normal
3,2.33,Rendah,94,Kurang,31,Cukup,17,Banyak
4,2.52,Sedang,138,Kurang,34,Cukup,10,Banyak
5,2.07,Rendah,123,Kurang,35,Cukup,19,Banyak
6,2.64,Sedang,141,Kurang,34,Cukup,5,Normal
7,2.96,Baik,130,Kurang,35,Cukup,5,Normal
8,2.64,Sedang,115,Kurang,33,Cukup,16,Banyak
9,2.54,Sedang,146,Cukup,36,Cukup,6,Banyak


## 5. Labeling Risiko DO (Drop Out)

Aturan labeling berdasarkan kondisi akademik:
- **DO**: semester > 14, IPK < 2.0, atau semester tidak aktif berturut > 2
- **Tinggi**: IPK 2.0–2.5 dengan semester ≥ 12, atau MK diulang banyak di semester awal
- **Sedang**: IPK 2.51–2.75 di semester menengah
- **Rendah**: kondisi di luar kriteria di atas

In [18]:
def label_risiko(row):
    # Kriteria DO (paling prioritas)
    if row['jumlah_semester'] > 14:
        return 'DO'
    if row['ipk'] < 2.0:
        return 'DO'
    if row['semester_tidak_aktif_berturut'] > 2:
        return 'DO'

    # Risiko Tinggi
    if (2.0 <= row['ipk'] <= 2.5 and row['jumlah_semester'] >= 12):
        return 'Tinggi'
    if row['jumlah_mk_diulang'] > 10:
        return 'Tinggi'

    # Risiko Sedang
    if (2.51 <= row['ipk'] <= 2.75 and 9 <= row['jumlah_semester'] <= 11):
        return 'Sedang'
    if row['jumlah_mk_diulang'] > 5:
        return 'Sedang'

    return 'Rendah'

df['risiko'] = df.apply(label_risiko, axis=1)
print('Distribusi label risiko:')
print(df['risiko'].value_counts())

Distribusi label risiko:
risiko
Rendah    22
Sedang    18
Tinggi    11
DO         4
Name: count, dtype: int64


In [19]:
# Cross-check dengan kolom status asli
print('Distribusi status asli:')
print(df['status'].value_counts())
print()
print('Crosstab status vs risiko:')
print(pd.crosstab(df['status'], df['risiko']))

Distribusi status asli:
status
A    51
T     4
Name: count, dtype: int64

Crosstab status vs risiko:
risiko  DO  Rendah  Sedang  Tinggi
status                            
A        2      20      18      11
T        2       2       0       0


In [20]:
# Tampilkan dataset final
df.head(10)

,jumlah_semester,ipk,total_sks_lulus,jumlah_mk_diulang,semester_terakhir_aktif,semester_tidak_aktif_berturut,jml_mk_wajib,jml_mk_pilihan_bebas_non_bidang,jml_mk_pilihan_bebas_prodi,jml_mk_pilihan_wajib_profil,status,kategori_ipk,kategori_sks,kategori_mk_wajib,kategori_mk_bebas_nb,kategori_mk_bebas_prodi,kategori_mk_wajib_profil,kategori_mk_diulang,risiko
0,11,3.03,147,6,20251,0,35,2,9,7,A,Baik,Cukup,Cukup,Cukup,Cukup,Cukup,Banyak,Sedang
1,11,2.93,134,5,20251,0,36,0,11,6,A,Baik,Kurang,Cukup,Kurang,Cukup,Cukup,Normal,Rendah
2,7,2.36,106,5,20231,4,32,2,3,6,T,Rendah,Kurang,Cukup,Cukup,Kurang,Cukup,Normal,DO
3,11,2.33,94,17,20251,0,31,1,3,5,A,Rendah,Kurang,Cukup,Kurang,Kurang,Kurang,Banyak,Tinggi
4,11,2.52,138,10,20251,0,34,0,4,12,A,Sedang,Kurang,Cukup,Kurang,Kurang,Cukup,Banyak,Sedang
5,11,2.07,123,19,20251,0,35,0,8,11,A,Rendah,Kurang,Cukup,Kurang,Cukup,Cukup,Banyak,Tinggi
6,11,2.64,141,5,20251,0,34,1,7,8,A,Sedang,Kurang,Cukup,Kurang,Cukup,Cukup,Normal,Sedang
7,11,2.96,130,5,20251,0,35,2,5,9,A,Baik,Kurang,Cukup,Cukup,Cukup,Cukup,Normal,Rendah
8,10,2.64,115,16,20251,0,33,2,7,5,A,Sedang,Kurang,Cukup,Cukup,Cukup,Kurang,Banyak,Tinggi
9,11,2.54,146,6,20251,0,36,2,7,12,A,Sedang,Cukup,Cukup,Cukup,Cukup,Cukup,Banyak,Sedang


## 6. Simpan Dataset

In [ ]:
#Forward chaining ga pake penanganan input

def check_risiko():
    print("--- Input Data Mahasiswa ---")
    ipk = float(input("IPK: "))
    smt = int(input("Semester: "))
    pa = int(input("Jumlah Peringatan Akademik: "))
    non_aktif = int(input("Status Non-Aktif (Smt berturut): "))
    ekonomi = input("Masalah Ekonomi/Keluarga? (y/n): ").lower()

    # Forward Chaining Singkat
    if ipk < 2.0 or smt > 14 or (ekonomi == 'y' and ipk <= 2.0):
        res = "TINGGI (POTENSI DO)"
    elif (2.0 <= ipk <= 2.5) or pa >= 2 or smt >= 12 or non_aktif >= 2:
        res = "TINGGI"
    elif (2.51 <= ipk <= 2.75) or pa == 1 or (9 <= smt <= 11) or ekonomi == 'y':
        res = "SEDANG"
    else:
        res = "RENDAH"

    print(f"\nKesimpulan: Risiko {res}")

check_risiko()

--- Input Data Mahasiswa ---

Kesimpulan: Risiko TINGGI (POTENSI DO)


In [ ]:
#Forward chaining pake penanganan input



def get_input_angka(prompt):
    """Fungsi pembantu untuk memastikan input adalah angka."""
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("❌ Input salah! Mohon masukkan angka (contoh: 2.5 atau 0).")

def check_risiko_robust():
    print("=== Sistem Deteksi Risiko Akademik (Input Aman) ===")
    
    # Input dengan validasi otomatis
    ipk = get_input_angka("Masukkan IPK: ")
    smt = int(get_input_angka("Masukkan Semester: "))
    pa = int(get_input_angka("Jumlah Peringatan Akademik: "))
    non_aktif = int(get_input_angka("Status Non-Aktif (Semester berturut): "))
    
    ekonomi = ""
    while ekonomi not in ['y', 'n']:
        ekonomi = input("Masalah Ekonomi/Keluarga? (y/n): ").lower()
        if ekonomi not in ['y', 'n']:
            print("❌ Mohon jawab dengan 'y' atau 'n'.")

    # Logika Forward Chaining (Aturan Kamu)
    # R1: Jika Fakta memenuhi syarat DO
    if ipk < 2.0 or smt > 14 or (ekonomi == 'y' and ipk <= 2.0):
        res = "TINGGI (POTENSI DO)"
    
    # R2: Jika R1 gagal, cek Risiko Tinggi
    elif (2.0 <= ipk <= 2.5) or pa >= 2 or smt >= 12 or non_aktif >= 2:
        res = "TINGGI"
    
    # R3: Jika R1 & R2 gagal, cek Risiko Sedang
    elif (2.51 <= ipk <= 2.75) or pa == 1 or (9 <= smt <= 11) or ekonomi == 'y':
        res = "SEDANG"
    
    # R4: Jika semua aturan di atas tidak terpenuhi
    else:
        res = "RENDAH"

    print("-" * 40)
    print(f"KESIMPULAN AKHIR: Risiko {res}")
    print("-" * 40)

check_risiko_robust()

=== Sistem Deteksi Risiko Akademik (Input Aman) ===
❌ Input salah! Mohon masukkan angka (contoh: 2.5 atau 0).
❌ Mohon jawab dengan 'y' atau 'n'.
----------------------------------------
KESIMPULAN AKHIR: Risiko TINGGI
----------------------------------------


In [22]:
df.to_csv('dataset_mhs_2020_2021_processed.csv', index=False)
print(f'Dataset berhasil disimpan: {df.shape[0]} baris, {df.shape[1]} kolom')
print('Kolom:', df.columns.tolist())

Dataset berhasil disimpan: 55 baris, 19 kolom
Kolom: ['jumlah_semester', 'ipk', 'total_sks_lulus', 'jumlah_mk_diulang', 'semester_terakhir_aktif', 'semester_tidak_aktif_berturut', 'jml_mk_wajib', 'jml_mk_pilihan_bebas_non_bidang', 'jml_mk_pilihan_bebas_prodi', 'jml_mk_pilihan_wajib_profil', 'status', 'kategori_ipk', 'kategori_sks', 'kategori_mk_wajib', 'kategori_mk_bebas_nb', 'kategori_mk_bebas_prodi', 'kategori_mk_wajib_profil', 'kategori_mk_diulang', 'risiko']
